# RNN (Recurrent Neural Network)

## 1. Purpose
RNNs are designed to process sequential data such as text, time series, and speech.
They maintain a hidden state that carries information from previous time steps.

## 2. Input Preparation
For text, convert tokens into integer IDs and pass them through an embedding layer.

input_ids shape: [B, L]
embedded shape: [B, L, E]

B = Batch size
L = Sequence length
E = Embedding dimension
H = Hidden size
V = Vocabulary size
C = Number of classes

## 3. Model Architecture
Embedding -> RNN -> Linear layer

Example:

embedded = self.embedding(input_ids)
output, h_n = self.rnn(embedded)
logits = self.fc(output)

For batch_first=True and a unidirectional RNN:

input_ids: [B, L]
embedded:  [B, L, E]
output:    [B, L, H]
h_n:       [num_layers, B, H]

## 4. RNN Outputs
output:
Contains the hidden state at every time step from the final RNN layer.

h_n:
Contains the final hidden state of each RNN layer.

## 5. Logits
Logits are raw scores produced by the final linear layer.
They are not probabilities.

For sequence classification:

final_hidden = h_n[-1]
logits = self.fc(final_hidden)

logits shape: [B, C]

For next-token prediction:

logits = self.fc(output)

logits shape: [B, L, V]

The linear layer converts hidden dimension H into the number of classes C
or vocabulary size V.

## 6. Loss Function

### Sequence Classification
logits shape: [B, C]
labels shape: [B]

criterion = nn.CrossEntropyLoss()
loss = criterion(logits, labels)

### Next-Token Prediction
Shift the input and target sequences by one token.

inputs = tokens[:, :-1]
targets = tokens[:, 1:]

logits shape: [B, L, V]
targets shape: [B, L]

Flatten the batch and sequence dimensions:

loss = criterion(
    logits.reshape(-1, logits.size(-1)),
    targets.reshape(-1)
)

For padded sequences:

criterion = nn.CrossEntropyLoss(ignore_index=PAD_ID)

Do not apply softmax before CrossEntropyLoss.

## 7. Optimizer
RNNs can use the same optimizers as CNNs.

Examples:
- Adam
- AdamW
- SGD

Example:

optimizer = torch.optim.Adam(model.parameters(), lr=0.001)

Gradient clipping can help control exploding gradients:

loss.backward()

torch.nn.utils.clip_grad_norm_(
    model.parameters(),
    max_norm=1.0
)

optimizer.step()

Gradient clipping is optional and is not an optimizer.

## 8. Training Loop Changes
The basic training loop remains the same as CNN training.

Main differences:
- Prepare sequential inputs.
- Use an RNN architecture.
- Select the appropriate hidden state or all time-step outputs.
- Prepare targets according to the task.
- Calculate the loss using the correct tensor shapes.
- Consider gradient clipping.

## 9. Backpropagation
RNNs use Backpropagation Through Time (BPTT).
Gradients flow through recurrent operations across time steps.
PyTorch handles this automatically through loss.backward().

## 10. Prediction
For sequence classification:

predictions = logits.argmax(dim=1)

For next-token prediction:

predictions = logits.argmax(dim=-1)

Sequence-classification predictions have shape [B].
Next-token predictions have shape [B, L].

| Architecture | Input      | Output     | Example                                   |
| ------------ | ---------- | ---------- | ----------------------------------------- |
| One-to-one   | One input  | One output | Ordinary image classification             |
| One-to-many  | One input  | A sequence | Generating a sequence from a single input |
| Many-to-one  | A sequence | One output | Sentiment classification                  |
| Many-to-many | A sequence | A sequence | Sequence labeling or translation          |


| Symbol      | Meaning                        |
| ----------- | ------------------------------ |
| \(x_t\)     | Input at the current time step |
| \(h_t\)     | Current hidden state           |
| \(h_{t-1}\) | Previous hidden state          |
| \(W_{xh}\)  | Input-to-hidden weights        |
| \(W_{hh}\)  | Hidden-to-hidden weights       |
| \(b_h\)     | Bias                           |
| \(\tanh\)   | Activation function            |


In [1]:
import torch
from torch import nn

In [2]:
x = torch.tensor([
    [[1.0, 2.0, 3.0],
     [4.0, 5.0, 6.0],
     [7.0, 8.0, 9.0],
     [10.0, 11.0, 12.0]],

    [[13.0, 14.0, 15.0],
     [16.0, 17.0, 18.0],
     [19.0, 20.0, 21.0],
     [22.0, 23.0, 24.0]]
])


In [3]:
x.shape

torch.Size([2, 4, 3])

In [4]:
rnn = nn.RNN(
    input_size=3,
    hidden_size=3,
    num_layers=1,
    batch_first=True
)

In [5]:
output, h_n = rnn(x)

In [6]:
output.shape,h_n.shape

(torch.Size([2, 4, 3]), torch.Size([1, 2, 3]))

In [7]:
print(output)
print()
print()
print(h_n)

tensor([[[ 0.3031, -0.9597, -0.1158],
         [ 0.9548, -0.9997, -0.2248],
         [ 0.9986, -1.0000, -0.1628],
         [ 1.0000, -1.0000, -0.3722]],

        [[ 1.0000, -1.0000, -0.8094],
         [ 1.0000, -1.0000, -0.7328],
         [ 1.0000, -1.0000, -0.8282],
         [ 1.0000, -1.0000, -0.8938]]], grad_fn=<TransposeBackward1>)


tensor([[[ 1.0000, -1.0000, -0.3722],
         [ 1.0000, -1.0000, -0.8938]]], grad_fn=<StackBackward0>)


In [8]:
print(output[:, -1, :].shape)
print(h_n[-1].shape)

torch.Size([2, 3])
torch.Size([2, 3])


In [9]:
print(rnn.weight_ih_l0)
print(rnn.weight_hh_l0)
print(rnn.bias_ih_l0)
print(rnn.bias_hh_l0)

Parameter containing:
tensor([[ 0.1139,  0.4241,  0.0654],
        [-0.5523, -0.0355, -0.2891],
        [ 0.0409, -0.3363,  0.2113]], requires_grad=True)
Parameter containing:
tensor([[ 0.0065,  0.1939,  0.4781],
        [-0.3907, -0.4014,  0.1870],
        [ 0.4965,  0.0036,  0.0619]], requires_grad=True)
Parameter containing:
tensor([-0.3881, -0.3917, -0.2029], requires_grad=True)
Parameter containing:
tensor([-0.4572, -0.0597,  0.0844], requires_grad=True)


In [10]:
rnn.weight_ih_l0
rnn.weight_hh_l0
rnn.bias_ih_l0
rnn.bias_hh_l0 

# 5 hidden units  → 5 values in this output

Parameter containing:
tensor([-0.4572, -0.0597,  0.0844], requires_grad=True)

# **Implementing**

In [11]:
X = torch.tensor([
    [[1.0], [2.0], [3.0], [4.0]],
    [[5.0], [4.0], [3.0], [-1.0]],
    [[2.0], [3.0], [4.0], [5.0]],
    [[5.0], [4.0], [3.0], [-2.0]]
])

y = torch.tensor([1, 0, 1, 0])

In [12]:
print(X.shape)
print(y.shape)

torch.Size([4, 4, 1])
torch.Size([4])


In [13]:
class SimpleRNN(nn.Module):
    def __init__(self):
        super().__init__()

        self.Rnn_layer = nn.RNN(
            input_size=1,
            hidden_size=8,
            num_layers=1,
            batch_first=True
        )

        self.linear = nn.Linear(8, 2)

    def forward(self, x):
        output, h_n = self.Rnn_layer(x)

        h_n = h_n[-1]

        output = self.linear(h_n)

        return output

In [14]:
model = SimpleRNN()

predict=model(X)

print(predict)
print(predict.shape)

tensor([[ 0.1145, -0.2086],
        [-0.3018, -0.1220],
        [ 0.1834, -0.2762],
        [-0.3750, -0.1061]], grad_fn=<AddmmBackward0>)
torch.Size([4, 2])


In [15]:
loss_fn = nn.CrossEntropyLoss()
optimizer = torch.optim.Adam(model.parameters(), lr=0.001)

In [16]:
epochs=5
for epoch in range(epochs):

    model.train()

    # forward pass
    pred = model(X)

    # calculate loss
    loss = loss_fn(pred, y)

    # clear old gradients
    optimizer.zero_grad()

    # calculate gradients
    loss.backward()

    # update parameters
    optimizer.step()

    if epoch % 10 == 0:
        print(f"{epoch} --> {loss.item()}")

0 --> 0.8601217269897461
